# 01 - Modelo de Marketing de Propensão a Campanhas Promocionais (SmartRetail)

Este notebook documenta o desenvolvimento, validação e interpretação de um **modelo de marketing de propensão a campanhas promocionais usando Árvore de Decisão**.

O objetivo de negócio é prever quais clientes ativos possuem maior probabilidade de responder positivamente a novas ofertas da **SmartRetail**, de modo que a equipe de marketing possa segmentar sua carteira de clientes, reduzindo custos de disparos ineficientes e aumentando a taxa de conversão final.

### Estrutura de Diretório Recomendada:
*   Entrada de Dados: `data/clientes_marketing.csv`
*   Pasta de Saída dos Gráficos: `plots/smartretail_decision_tree.png`

## 1. Carregamento de Dependências do Projeto

Importamos as ferramentas necessárias para manipulação estruturada dos dados de consumo e para o treinamento do classificador preditivo de propensão baseados na Árvore de Decisão.

In [ ]:
import logging
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Componentes de modelagem e validação de propensão do Scikit-Learn
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Definições estéticas para os plots das métricas de marketing preditivo
sns.set_theme(style="whitegrid")

## 2. Carga e Diagnóstico de Qualidade dos Dados da SmartRetail

Realizamos a leitura do histórico de marketing e comportamento dos clientes a partir do arquivo localizado de forma isolada na subpasta `data`.

In [ ]:
caminho_dados = Path("data") / "clientes_marketing.csv"

# Verificação de segurança de leitura
if not caminho_dados.exists():
    logging.error(f"Erro: Arquivo do histórico de campanhas não localizado em {caminho_dados.resolve()}")
else:
    df = pd.read_csv(caminho_dados)
    print(f"Dataset SmartRetail carregado. Amostras: {df.shape[0]} | Variáveis: {df.shape[1]}\n")
    display(df.head())

In [ ]:
# Avaliação estrutural dos metadados e integridade (presença de nulos)
df.info()

## 3. Particionamento e Estratificação dos Dados

Isolamos a variável alvo do modelo (`RespondeuCampanha`) das características dos clientes. Em seguida, dividimos o conjunto em **80% para treinamento** e **20% para validação independente (teste)**, de forma estratificada para manter o exato balanceamento das respostas de campanha.

In [ ]:
X = df.drop(columns=['RespondeuCampanha'])
y = df['RespondeuCampanha']

# A estratificação garante que as proporções de resposta das campanhas se mantenham corretas em treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Amostras de Treino (ajuste do modelo de propensão): {X_train.shape[0]}")
print(f"Amostras de Teste (validação do modelo de propensão): {X_test.shape[0]}")

## 4. Treinamento do Modelo de Propensão a Campanhas Promocionais

Treinamos o modelo de Árvore de Decisão limitando a sua profundidade máxima (`max_depth=3`) para assegurar que suas decisões de segmentação de marketing sejam simples e auditáveis pelas áreas de negócio da SmartRetail.

In [ ]:
modelo_dt = DecisionTreeClassifier(random_state=42, max_depth=3)
modelo_dt.fit(X_train, y_train)

print("Árvore de Decisão de Propensão à Campanha treinada com sucesso.")

## 5. Avaliação do Desempenho do Modelo de Marketing

Geramos as predições de propensão de conversão para o conjunto de validação independente (teste) e avaliamos os acertos do modelo por meio de uma matriz de confusão estilizada.

In [ ]:
# Geração de previsões de propensão
y_pred_dt = modelo_dt.predict(X_test)
acuracia = accuracy_score(y_test, y_pred_dt)

print(f"Acurácia Geral do Classificador de Propensão: {acuracia * 100:.2f}%\n")

# Gráfico da Matriz de Confusão para Visualização dos Acertos e Desvios do Modelo
matriz = confusion_matrix(y_test, y_pred_dt)
plt.figure(figsize=(6, 4))
sns.heatmap(
    matriz, 
    annot=True, 
    fmt='d', 
    cmap='Oranges', 
    xticklabels=['Não Respondeu (0)', 'Respondeu (1)'], 
    yticklabels=['Não Respondeu (0)', 'Respondeu (1)']
)
plt.title('Matriz de Confusão - Modelo de Propensão SmartRetail')
plt.ylabel('Resposta Real à Campanha')
plt.xlabel('Predição de Propensão')
plt.show()

# Relatório Estatístico Detalhado de Precisão e Cobertura (Recall)
print("\nRelatório de Classificação Detalhado:")
print(classification_report(y_test, y_pred_dt))

## 6. Visualização do Diagrama de Decisão de Propensão

Abaixo, renderizamos o fluxo de decisões gerado pela Árvore de Decisão. Esse fluxograma estabelece os critérios ideais de corte para a nossa régua de segmentação de marketing promocional.

In [ ]:
plt.figure(figsize=(14, 8))
plot_tree(
    modelo_dt, 
    feature_names=X.columns, 
    class_names=['Não Respondeu (0)', 'Respondeu (1)'], 
    filled=True, 
    rounded=True, 
    fontsize=10
)
plt.title("Fluxograma de Decisões de Classificação - Modelo de Propensão SmartRetail", fontsize=14, pad=15)
plt.show()

## 7. Conclusões de Negócio e Próximos Passos

*   **Desempenho Geral:** O classificador de propensão obteve uma acurácia de 100,00% na partição de testes. Esse resultado é característico da estrutura linear limpa e de tamanho simplificado (50 observações) do conjunto de testes fornecido.
*   **Aplicação Prática da Regra:** O modelo aponta de forma transparente que a SmartRetail deve priorizar campanhas promocionais focadas em clientes com gasto anual acumulado acima de R$ 1.850,00, minimizando o custo com clientes de menor engajamento.
*   **Decisão Estratégica:** A Árvore de Decisão constitui-se como o modelo mais viável para o CRM da SmartRetail, pois gera regras legíveis de fácil validação pelas equipes comerciais da empresa, dispensando etapas de escala de dados em produção.